# 🐾 22 · Object Tracking
### Linking detections across time into trajectories — from Bayesian filters to the Hungarian algorithm, lineage recovery, and optical flow

**Part G · Across time and space** · [⟵ 20 Object detection](20_object_detection.ipynb) · [🏠 Course home](../README.md) · [23 Image registration ⟶](23_image_registration.ipynb)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/22_object_tracking.ipynb)

| | |
|---|---|
| **Hardware target** | CPU, a laptop is enough (< 1 min to run end to end) |
| **Core frameworks** | NumPy, SciPy (`scipy.optimize.linear_sum_assignment`), Matplotlib |
| **Prerequisites** | Module 13 (keypoints), Module 18 (segmentation), Module 20 (detection) |
| **Estimated time** | 100–130 minutes |
| **02-740 topics covered** | contour-evolution tracking, particle filters, tracking-by-detection, the linear assignment problem, the Hungarian algorithm, cell lineage, segment-level assignment and lineage recovery, Lucas-Kanade optical flow, the aperture problem |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> All images, time-lapses and ground-truth tracks in this notebook are SIMULATED by code — nothing is downloaded.


## 🎯 Learning objectives

By the end of this notebook you will be able to:

- explain why running an object detector independently on every frame of a movie does **not** give you tracking, and name the three concrete failure modes;
- describe the three broad families of bioimage tracking algorithms — contour evolution, particle filters, and tracking-by-detection — and when each is the right tool;
- derive and implement a simple **particle filter** as an instance of Bayesian recursive state estimation, and relate it to the **Kalman filter** as its linear-Gaussian special case;
- formulate frame-to-frame object linking as a **linear assignment problem (LAP)** and solve it with the **Hungarian algorithm**;
- build a complete **tracking-by-detection** pipeline — detect, link, evaluate — on a simulated dividing-cell time-lapse, and explain why a one-to-one frame-to-frame assignment loses information at cell divisions;
- extend frame-to-frame linking into a **segment-level linear assignment problem** that recovers gap-closing and splitting events (e.g. cell divisions) a single frame-to-frame LAP necessarily loses;
- implement **Lucas-Kanade optical flow** from image gradients to estimate motion directly, without any detection step, and explain the **aperture problem** that limits where it can be solved.


## 🗺️ Contents

1. Why detection-per-frame isn't tracking
2. Tracking by contour evolution (snakes revisited)
3. Particle filters: Bayesian state tracking
4. Tracking-by-detection and the linear assignment problem
5. From one-to-one assignment to lineages: gap closing, merging, splitting
6. Tracking without detection: Lucas-Kanade optical flow for collective motion


## 🧫 The biological question

A microscope gives you a *sequence* of snapshots, but biology happens in between the frames. A migrating immune cell, a dividing stem cell, a motor-driven vesicle, an elongating actin filament — in every one of these cases the scientific question is not "where is the object in this one frame?" but "how did this specific object's state evolve over time?" **Object tracking** is the task of recovering that continuity: assigning a consistent identity to the same physical object across a sequence of frames, so that you can later ask how fast it moved, whether it changed direction, when it divided, or how its shape deformed.

Cell motility (the spontaneous movement of a cell from place to place) and cell migration (its *directed* movement in response to a chemical or mechanical cue) are central to wound healing, immune surveillance, embryonic development and cancer metastasis — and every one of these phenomena is *invisible* to a single still image. Tracking is what turns a stack of pictures into a measurement of a dynamical process.

---
## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
try:
    import torch
    print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
except ImportError:
    print('PyTorch not installed yet — the next cell installs it if needed.')


In [ ]:
!pip install -q scikit-image
# ── Dependencies ── (Colab already has most of these; we install only what is missing)


In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
SEED = 14
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11


### 🧰 Course toolkit

We reuse `simulate_time_lapse` (a moving, occasionally dividing population of blobs with known ground-truth tracks and lineage), `hungarian_assignment` (the linear assignment solver), `ConstantVelocityKalman` (a from-scratch Kalman filter), `chan_vese_step` (the active-contour update from Module 18, reused here for contour-evolution tracking) and two plotting helpers.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

def _draw_blob(canvas: np.ndarray, cy: float, cx: float, r: float, shape: str,
                rng: np.random.Generator, angle: float = 0.0) -> None:
    """Add one synthetic nucleus/cell of the given shape class onto ``canvas`` (in place, additive)."""
    size = canvas.shape[0]
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    y, x = yy - cy, xx - cx
    ca, sa = np.cos(angle), np.sin(angle)
    yr, xr = ca * y + sa * x, -sa * y + ca * x  # rotate into the blob's own frame
    if shape == "round":
        d = (yr / r) ** 2 + (xr / r) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "elongated":
        d = (yr / (r * 0.55)) ** 2 + (xr / (r * 1.7)) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "kidney":
        d = (yr / r) ** 2 + (xr / r) ** 2
        dent = np.exp(-((yr - 0.0) ** 2 + (xr - r * 0.9) ** 2) / (2 * (r * 0.5) ** 2))
        canvas += np.clip(np.exp(-d * 2.0) - 0.9 * dent, 0, None)
    elif shape == "ring":
        d = np.sqrt(yr ** 2 + xr ** 2)
        canvas += np.exp(-((d - r) ** 2) / (2 * (r * 0.28) ** 2))
    elif shape == "dividing":
        sep = r * 1.1
        for sign in (-1, 1):
            d = ((yr - sign * sep * 0.5) / (r * 0.75)) ** 2 + (xr / (r * 0.75)) ** 2
            canvas += np.exp(-d * 2.0)
    else:  # irregular: a round blob perturbed by a few random bumps
        d = (yr / r) ** 2 + (xr / r) ** 2
        base = np.exp(-d * 2.0)
        for _ in range(3):
            bx, by = rng.uniform(-r, r, size=2)
            bump = np.exp(-(((yr - by) ** 2 + (xr - bx) ** 2) / (2 * (r * 0.45) ** 2)))
            base = np.clip(base + 0.5 * bump, 0, 1.3)
        canvas += base


def add_gaussian_noise(image: np.ndarray, sigma: float, rng: np.random.Generator) -> np.ndarray:
    """Additive Gaussian (read) noise: the classic noise floor of a camera sensor."""
    return image + rng.normal(0.0, sigma, size=image.shape)


def blur_image(image: np.ndarray, psf: np.ndarray) -> np.ndarray:
    """Convolve an image with a point-spread function (microscope optical blur)."""
    return ndi.convolve(image, psf, mode="reflect")


def gaussian_psf(sigma: float, radius: int | None = None) -> np.ndarray:
    """A normalised 2-D Gaussian point-spread function (the microscope's blur kernel)."""
    if radius is None:
        radius = max(1, int(np.ceil(3 * sigma)))
    ax = np.arange(-radius, radius + 1)
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    return k / k.sum()


def simulate_time_lapse(n_frames: int = 20, n_particles: int = 12, size: int = 160,
                         speed: float = 2.5, division_prob: float = 0.03,
                         seed: int = 0):
    """A SIMULATED time-lapse of moving, occasionally dividing particles/cells.

    Returns ``frames`` (list of n_frames images), ``tracks`` (dict: true track id -> list of
    (frame, y, x)), and ``lineage`` (dict: child track id -> parent track id, for divisions).
    """
    rng = np.random.default_rng(seed)
    margin = size * 0.1
    positions = {i: np.array([rng.uniform(margin, size - margin), rng.uniform(margin, size - margin)])
                 for i in range(n_particles)}
    velocities = {i: rng.normal(0, speed, size=2) for i in positions}
    tracks = {i: [] for i in positions}
    lineage = {}
    next_id = n_particles
    frames = []
    for t in range(n_frames):
        canvas = np.full((size, size), 0.05)
        for pid in list(positions):
            positions[pid] = np.clip(positions[pid] + velocities[pid], margin, size - margin)
            for k in (0, 1):
                if positions[pid][k] in (margin, size - margin):
                    velocities[pid][k] *= -1
            _draw_blob(canvas, *positions[pid], size * 0.03, "round", rng)
            tracks[pid].append((t, *positions[pid]))
            if t > 2 and rng.random() < division_prob and len(positions) < n_particles * 3:
                child = next_id
                next_id += 1
                positions[child] = positions[pid] + rng.normal(0, 3, size=2)
                velocities[child] = -velocities[pid] + rng.normal(0, speed * 0.3, size=2)
                tracks[child] = []
                lineage[child] = pid
        canvas = blur_image(canvas, gaussian_psf(1.1))
        frames.append(add_gaussian_noise(canvas, 0.02, rng).clip(0, None))
    return dict(frames=frames, tracks=tracks, lineage=lineage, source="SIMULATED")


def hungarian_assignment(cost_matrix: np.ndarray):
    """The Hungarian algorithm for the linear assignment problem: the lowest-total-cost one-to-
    one matching between rows and columns of ``cost_matrix``. Returns (row_ind, col_ind)."""
    return linear_sum_assignment(cost_matrix)


class ConstantVelocityKalman:
    """A minimal 2-D constant-velocity Kalman filter for one tracked point (state = y, x, vy, vx)."""

    def __init__(self, y0: float, x0: float, process_var: float = 1.0, meas_var: float = 4.0):
        self.x = np.array([y0, x0, 0.0, 0.0])
        self.P = np.eye(4) * 10.0
        self.F = np.array([[1, 0, 1, 0], [0, 1, 0, 1], [0, 0, 1, 0], [0, 0, 0, 1]], dtype=float)
        self.H = np.array([[1, 0, 0, 0], [0, 1, 0, 0]], dtype=float)
        self.Q = np.eye(4) * process_var
        self.R = np.eye(2) * meas_var

    def predict(self):
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
        return self.x[:2].copy()

    def update(self, measurement: np.ndarray):
        y = measurement - self.H @ self.x
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        self.x = self.x + K @ y
        self.P = (np.eye(4) - K @ self.H) @ self.P
        return self.x[:2].copy()


def chan_vese_step(image: np.ndarray, phi: np.ndarray, mu: float = 0.2, dt: float = 0.5):
    """One explicit-Euler update of the Chan–Vese active-contour (level-set) evolution: moves
    the zero level set of ``phi`` to better separate the two regions of (near-)constant
    intensity it currently implies. Returns the updated ``phi``."""
    inside = phi > 0
    c1 = image[inside].mean() if inside.any() else image.mean()
    c2 = image[~inside].mean() if (~inside).any() else image.mean()
    grad_y, grad_x = np.gradient(phi)
    grad_mag = np.hypot(grad_y, grad_x) + 1e-6
    kappa = (np.gradient(grad_y / grad_mag, axis=0) + np.gradient(grad_x / grad_mag, axis=1))
    force = -(image - c1) ** 2 + (image - c2) ** 2
    return phi + dt * grad_mag * (mu * kappa + force)


def show_image(ax, image: np.ndarray, title: str = "", cmap: str = "gray"):
    """Show one image on a given Matplotlib axis, titled, with ticks removed."""
    ax.imshow(image, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])


def show_image_grid(images, titles=None, ncols: int = 4, cmap: str = "gray", figsize_scale: float = 2.6):
    """A grid of images with Matplotlib, one subplot each, titles optional."""
    n = len(images)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_scale * ncols, figsize_scale * nrows))
    axes = np.atleast_1d(axes).ravel()
    for i, img in enumerate(images):
        show_image(axes[i], img, titles[i] if titles else "", cmap=cmap)
    for ax in axes[n:]:
        ax.axis("off")
    fig.tight_layout()
    return fig, axes


def plot_tracks(ax, tracks: dict, min_len: int = 2):
    """Plot every track in a ``{id: [(t, y, x), ...]}`` dict as a coloured polyline."""
    for tid, pts in tracks.items():
        if len(pts) < min_len:
            continue
        arr = np.array(pts)
        ax.plot(arr[:, 2], arr[:, 1], marker="o", markersize=2, linewidth=1)
    ax.invert_yaxis()
    ax.set_aspect("equal")

---

## 1 · Why detection-per-frame isn't tracking

Suppose you already have a perfect object detector (Module 20) and you run it independently on every frame of a movie. Is that tracking? No — and the gap is not a minor technicality. Three concrete problems appear the moment you have more than a single, lone object:

1. **The linking problem.** If a frame contains several objects, independent per-frame detection gives you an *unordered set* of boxes or points in each frame — it tells you nothing about *which* object in frame $t$ corresponds to *which* object in frame $t+1$. Without that correspondence you cannot compute a single velocity, a single trajectory, or a single division event.
2. **Appearance and disappearance.** Real objects drift out of the field of view, get temporarily occluded, or fall below the detection threshold for a few frames and then reappear. A detector alone has no way to decide whether a newly-detected object is *the same* one that vanished three frames ago or a genuinely new one.
3. **No motion information.** A detector looks at one frame at a time, so it can never report a velocity, an acceleration, or a direction — quantities that are often the entire scientific point (e.g. is this immune cell moving faster toward the wound than that one?).

In [ ]:
# ── 1.1 Demonstrate the problem: per-frame detection gives you unordered point clouds ──
sim = simulate_time_lapse(n_frames=20, n_particles=10, size=160, speed=2.2,
                           division_prob=0.025, seed=SEED)
frames = sim["frames"]

def detect_frame_blobs(frame, z_thresh=3.0, min_area=4):
    """A minimal from-scratch blob detector: threshold at (mean + z_thresh * std), label
    connected components, return one centroid per component. This notebook's own detector —
    deliberately simple so the tracking logic, not the detector, is the point."""
    thresh = frame.mean() + z_thresh * frame.std()
    mask = frame > thresh
    labeled, n = ndi.label(mask)
    centroids = []
    for i in range(1, n + 1):
        ys, xs = np.where(labeled == i)
        if len(ys) >= min_area:
            centroids.append((ys.mean(), xs.mean()))
    return np.array(centroids) if centroids else np.zeros((0, 2))

detections_per_frame = [detect_frame_blobs(f) for f in frames]
fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
for ax, t in zip(axes, (0, 1, 2)):
    show_image(ax, frames[t], f"frame {t}: {len(detections_per_frame[t])} detections")
    pts = detections_per_frame[t]
    if len(pts):
        ax.scatter(pts[:, 1], pts[:, 0], s=14, c="red")
fig.tight_layout()
print("Detections per frame (first 5):", [len(d) for d in detections_per_frame[:5]])
print("Nothing above tells you which red dot in frame 0 became which red dot in frame 1.")

🔍 **What just happened?** We ran the same simple detector on every frame independently. Each frame gives us a *cloud* of (y, x) points with **no persistent identity attached** — frame 1's third point has no label connecting it to frame 0's points. Counting objects per frame even fluctuates slightly as detections appear/disappear near threshold. Tracking is precisely the missing step: turning this sequence of point clouds into a set of trajectories.

✏️ **Your turn.** Lower `z_thresh` in `detect_frame_blobs` to `1.5` and rerun. More spurious detections appear from noise. Which of the three problems above (linking / appearance-disappearance / no motion info) does this make *worse*, and which does it leave unchanged? Write one sentence for each.

## 2 · Tracking by contour evolution: snakes revisited

Module 18 introduced the **active contour (snake)**: a parametric curve $v(s) = (x(s), y(s))$ that deforms to minimise an energy functional

$$E_{\text{snake}} = \int_0^1 \Big[\underbrace{E_{\text{int}}(v(s))}_{\text{smoothness}} + \underbrace{E_{\text{image}}(v(s))}_{\text{pulled toward edges}} + \underbrace{E_{\text{con}}(v(s))}_{\text{user constraints}}\Big]\,ds .$$

*In words:* the internal energy keeps the curve smooth and continuous, the image energy pulls it toward strong gradients (object boundaries), and an optional constraint energy lets a user nudge it. For **tracking**, the key trick is simple and elegant: **the converged contour from frame $k$ is used as the initial contour for frame $k+1$.** Because real biological structures move only a little between consecutive frames, the previous frame's answer is already close to the new frame's boundary, so the snake only has to adapt — grow, shrink, or drift — rather than search from scratch.

This is exactly how **actin filament tracking** has been approached (Saban et al. 2006, Hadjidemetriou et al. 2008, Li et al. 2009): actin filaments are thin (~7 nm), 10 µm structures whose tracking needs to follow both the filament *body* and its two *tips* as they grow, shrink, and bend. Real implementations also need a tip-finding step (often combined with a particle filter — see Section 3) because classical snakes have a narrow "capture range": the contour must already be close to the true boundary to converge, and the model struggles when a filament's intensity briefly dims (under-growth) or two nearby tips get confused (over-growth).

In [ ]:
# ── 2.1 Contour evolution across frames: reuse the converged contour as next frame's init ──
size = 120
yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")

def make_moving_blob(cy, cx, r=18):
    img = np.zeros((size, size))
    d = (yy - cy) ** 2 + (xx - cx) ** 2
    img += np.exp(-d / (2 * r ** 2))
    return add_gaussian_noise(img, 0.03, rng)

track_frames = [make_moving_blob(60 + 3 * t, 40 + 4 * t) for t in range(6)]
phi = (np.sqrt((yy - 60) ** 2 + (xx - 40) ** 2) < 22).astype(float) * 2 - 1  # initial contour (frame 0)

converged_contours = []
for t, frame in enumerate(track_frames):
    for _ in range(60):  # a handful of explicit-Euler steps per frame is enough to re-converge
        phi = chan_vese_step(frame, phi, mu=0.15, dt=0.4)
    converged_contours.append(phi.copy())
    # KEY IDEA: do NOT reset phi for the next frame — the converged contour becomes next frame's init

fig, axes = plt.subplots(1, 6, figsize=(14, 2.6))
for t, (ax, frame, phi_t) in enumerate(zip(axes, track_frames, converged_contours)):
    show_image(ax, frame, f"t={t}")
    ax.contour(phi_t, levels=[0], colors="lime", linewidths=1.5)
fig.tight_layout()

🔍 **What just happened?** We never re-initialised the contour from scratch. Each frame's converged level set `phi` became the *starting* level set for the next frame's Chan–Vese updates (Module 18's building block, reused here with no modification). Because the blob only moved a few pixels between frames, a handful of update steps were enough for the contour to "follow" it — the same principle that lets a snake track a growing or shrinking actin filament frame by frame.

✏️ **Your turn.** Increase the step size (`4 * t` → `10 * t` in `make_moving_blob`) so the blob jumps farther between frames. At some point the contour will fail to keep up and get left behind. This is the snake's **limited capture range** in action — the exact weakness noted for traditional snakes (sensitivity to initial position, small capture range), which motivated the gradient-vector-flow (GVF) extension.

## 3 · Particle filters: tracking as Bayesian state estimation

Particle filters were originally developed for **robot localisation**: a robot with distance sensors moves through a known map and must estimate its own position from noisy range measurements. The same mathematical machinery tracks a moving cell or particle — "position in a city map" becomes "position in an image," and "distance sensor readings" become "local image intensity around a candidate position."

*In words:* a particle filter represents a belief about an object's state (e.g. its position) not as a single number but as a **cloud of weighted hypotheses** ("particles"). At every new frame it alternates two steps dictated by Bayes' rule applied recursively in time:

$$p(x_t \mid z_{1:t}) \;\propto\; \underbrace{p(z_t \mid x_t)}_{\text{likelihood: does this particle explain the new measurement?}} \int \underbrace{p(x_t \mid x_{t-1})}_{\text{motion model}}\, p(x_{t-1}\mid z_{1:t-1})\, dx_{t-1}$$

Concretely, the robot-localisation recipe (and its tracking analogue) is:

1. **Predict / propagate.** Move every particle according to a motion model (plus random jitter), representing "the object probably moved like this, plus or minus some uncertainty."
2. **Weight.** Compare each particle's predicted position against the actual new measurement (e.g. local image intensity, or a feature-similarity score against a template) and assign it a likelihood weight — particles that better explain the data get higher weight.
3. **Resample.** Draw a new particle cloud with probability proportional to the weights, so that promising hypotheses are explored more densely and poor ones are dropped. Over a few iterations the cloud **converges** onto the true state.

When the motion model and the measurement noise are both linear and Gaussian, the whole particle cloud can be replaced, *exactly*, by tracking only its mean and covariance — this is the **Kalman filter**, a much cheaper special case. `course_toolkit.ConstantVelocityKalman` is exactly that: a 4-state ($y, x, v_y, v_x$) linear-Gaussian filter. A particle filter is strictly more general (it handles multi-modal beliefs, non-linear motion, non-Gaussian noise) at the cost of needing many particles to be accurate.

In [ ]:
# ── 3.1 A from-scratch particle filter tracking one moving, noisy blob ──
def particle_filter_track(frames, init_state, n_particles=300, motion_std=3.0,
                           meas_std=6.0, seed=0):
    """Track the (y, x) position of one bright blob across `frames` with a particle filter.
    Returns the per-frame posterior-mean estimate (n_frames x 2)."""
    local_rng = np.random.default_rng(seed)
    particles = init_state + local_rng.normal(0, 10, size=(n_particles, 2))
    weights = np.full(n_particles, 1.0 / n_particles)
    estimates = []
    for frame in frames:
        # 1. Predict: propagate each particle with a random-walk motion model
        particles = particles + local_rng.normal(0, motion_std, size=(n_particles, 2))
        # 2. Weight: how bright is the image at (around) each particle's position?
        iy = np.clip(particles[:, 0].round().astype(int), 0, frame.shape[0] - 1)
        ix = np.clip(particles[:, 1].round().astype(int), 0, frame.shape[1] - 1)
        measurement = frame[iy, ix]
        weights = np.exp(-0.5 * ((1.0 - measurement) / (meas_std / 20)) ** 2) + 1e-9
        weights /= weights.sum()
        estimates.append((particles * weights[:, None]).sum(axis=0))
        # 3. Resample proportional to weight
        idx = local_rng.choice(n_particles, size=n_particles, p=weights)
        particles = particles[idx]
    return np.array(estimates)

# Build a short single-blob time-lapse with a KNOWN ground-truth path to check against
single = simulate_time_lapse(n_frames=25, n_particles=1, size=160, speed=2.0,
                              division_prob=0.0, seed=SEED + 1)
true_path = np.array([(y, x) for (_, y, x) in single["tracks"][0]])

pf_estimate = particle_filter_track(single["frames"], init_state=true_path[0], seed=SEED)

kf = ConstantVelocityKalman(*true_path[0])
kf_estimate = []
for frame in single["frames"]:
    pred = kf.predict()
    # measure with the SAME simple blob detector as Section 1/4 — a fair comparison to the
    # particle filter, which also only ever sees raw pixels, never the ground truth
    found = detect_frame_blobs(frame, z_thresh=2.5, min_area=3)
    if len(found):
        nearest = found[np.argmin(np.hypot(found[:, 0] - pred[0], found[:, 1] - pred[1]))]
        kf_estimate.append(kf.update(nearest))
    else:
        kf_estimate.append(pred)  # no detection this frame: fall back on the motion-only prediction
kf_estimate = np.array(kf_estimate)

pf_error = np.hypot(*(pf_estimate - true_path).T).mean()
kf_error = np.hypot(*(kf_estimate - true_path).T).mean()
print(f"Particle filter mean position error: {pf_error:.2f} px")
print(f"Kalman filter   mean position error: {kf_error:.2f} px")

🧪 **Ground-truth check.** `simulate_time_lapse` returns the *true* (frame, y, x) path in `tracks` — we never showed it to either filter, only used it to initialise and to score. Both filters should track the blob to within a few pixels on average; the particle filter uses only raw image brightness as its "sensor," which is noisier than the Kalman filter's idealised constant-velocity assumption, so don't be surprised if the Kalman filter does slightly better *for this particular, well-behaved, constant-velocity motion* — that gap would close (or reverse) if the object's motion were erratic or multi-modal (e.g. it could be in one of two places), which is exactly where particle filters earn their extra cost.

### 📘 Lecture notes — Bayesian state estimation: particle filters and Kalman filters

Both filters are instances of the same recursive-Bayesian idea (predict with a motion model, then correct with a measurement), differing only in *how* they represent the belief distribution: a Kalman filter keeps a single Gaussian (mean + covariance) and updates it with closed-form linear algebra, which is exact when the true dynamics are linear-Gaussian and extremely cheap to compute. A particle filter keeps an entire *cloud* of weighted samples, which costs more but needs no linearity or Gaussian assumption, and naturally represents multi-modal beliefs (e.g. "the cell is probably here, but possibly over there"). In bioimage tracking, Kalman-style filters are the default when object motion is smooth, while particle filters (or their descendants) are preferred for erratic, bio-driven motion such as a vesicle undergoing intermittent active transport.

✏️ **Your turn.** Re-run `particle_filter_track` with `n_particles=20` instead of 300. Does the mean position error get noticeably worse? Particle filters trade particle count for accuracy — too few particles and the cloud can "collapse" onto the wrong mode.

## 4 · Tracking-by-detection and the linear assignment problem

**Tracking-by-detection** splits the problem cleanly into two stages: (1) detect objects independently in every frame (Section 1's weakness is a *feature*, not a bug, here — we deliberately decouple detection from linking), then (2) solve a **linking / association** problem that decides which detection in frame $t$ corresponds to which detection in frame $t+1$.

The simplest, most important formalisation of the linking step is the **linear assignment problem (LAP)**: given $n$ "workers" (detections in frame $t$) and $m$ "jobs" (detections in frame $t+1$), and a cost $c_{ij}$ for assigning worker $i$ to job $j$ (typically their distance, or a combination of distance and intensity similarity), find the one-to-one assignment that **minimises total cost**. Enumerating every possible assignment is combinatorially explosive ($n!$ for $n=n$ workers/jobs): even a toy 3×3 example has only 6 candidate assignments, but a 20×20 frame-to-frame link already has $20! \approx 2.4\times 10^{18}$ candidates.

The **Hungarian algorithm** (Kuhn 1955, building on Kőnig and Egerváry; sometimes called Flood's technique) solves the LAP exactly in polynomial time by (1) subtracting the row minima, (2) subtracting the column minima, (3) covering all the resulting zeros with the minimum number of lines, and (4) if that number is less than $n$, adjusting the matrix and repeating — until a feasible zero-cost assignment can be read off directly. `course_toolkit.hungarian_assignment` wraps SciPy's implementation of exactly this algorithm.

In [ ]:
# ── 4.1 Sanity check: reproduce a textbook 3-worker/3-job assignment problem ──
jobs_cost = np.array([
    [53, 96, 37],
    [47, 87, 41],
    [60, 92, 36],
], dtype=float)
rows, cols = hungarian_assignment(jobs_cost)
total_cost = jobs_cost[rows, cols].sum()
print("Optimal assignment (worker -> job):", list(zip(rows, cols)))
print("Total cost:", total_cost, "(the textbook minimum over all 6 possible assignments is 176)")
assert total_cost == 176, 'Hungarian algorithm should recover the known-minimum assignment'

🔍 **What just happened?** With 3 workers and 3 jobs there are only $3! = 6$ possible one-to-one assignments, so brute force is feasible and we can check the answer by hand: the costs of all 6 assignments are 176, 186, 179, 176, 197, 184 — so 176 is indeed the minimum, and the Hungarian algorithm found it without ever enumerating all 6. For frame-to-frame particle linking the same algorithm scales to hundreds of objects per frame in milliseconds.

In [ ]:
# ── 4.2 A full tracking-by-detection pipeline on a dividing-cell time-lapse ──
sim = simulate_time_lapse(n_frames=20, n_particles=10, size=160, speed=2.2,
                           division_prob=0.03, seed=SEED)
frames = sim["frames"]
detections_per_frame = [detect_frame_blobs(f) for f in frames]

GATE = 15.0  # px; a candidate link costing more than this is treated as "impossible"
next_track_id = 0
active = {}          # track_id -> last (y, x)
estimated_tracks = {}  # track_id -> list of (t, y, x)

for t, dets in enumerate(detections_per_frame):
    if not active:
        for y, x in dets:
            active[next_track_id] = (y, x)
            estimated_tracks[next_track_id] = [(t, y, x)]
            next_track_id += 1
        continue
    active_ids = list(active)
    prev_pts = np.array([active[i] for i in active_ids])
    if len(dets) == 0:
        active = {}
        continue
    cost = np.hypot(prev_pts[:, 0:1] - dets[:, 0], prev_pts[:, 1:2] - dets[:, 1])
    rows, cols = hungarian_assignment(cost)
    matched_dets = set()
    new_active = {}
    for r, c in zip(rows, cols):
        if cost[r, c] <= GATE:
            tid = active_ids[r]
            new_active[tid] = tuple(dets[c])
            estimated_tracks[tid].append((t, *dets[c]))
            matched_dets.add(c)
    # unmatched detections start new tracks (this is where divisions show up!)
    for c in range(len(dets)):
        if c not in matched_dets:
            new_active[next_track_id] = tuple(dets[c])
            estimated_tracks[next_track_id] = [(t, *dets[c])]
            next_track_id += 1
    active = new_active

print(f"Ground truth : {len(sim['tracks'])} true track ids, {len(sim['lineage'])} division event(s)")
print(f"Our pipeline : {len(estimated_tracks)} estimated track ids after linking")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
plot_tracks(axes[0], sim["tracks"], min_len=2)
axes[0].set_title("Ground truth tracks")
plot_tracks(axes[1], estimated_tracks, min_len=2)
axes[1].set_title("Hungarian tracking-by-detection")
fig.tight_layout()

🧪 **Ground-truth check.** Count how many estimated track ids were created versus the true number of tracks (`len(sim['tracks'])`, which already includes every division's child as a *new* true id). If our pipeline created noticeably *more* estimated ids than true ids, inspect which frames those extra ids started on — they should cluster around the frames where `sim['lineage']` records a division.

In [ ]:
# ── 4.3 Quantify: do new estimated tracks start where true divisions happened? ──
division_frames = set()
true_tracks = sim["tracks"]
for child_id, parent_id in sim["lineage"].items():
    if true_tracks[child_id]:
        division_frames.add(true_tracks[child_id][0][0])

new_track_start_frames = [v[0][0] for v in estimated_tracks.values() if v[0][0] > 0]
near_division = sum(any(abs(t - d) <= 1 for d in division_frames) for t in new_track_start_frames)
print(f"{len(division_frames)} true division events across the movie")
print(f"{len(new_track_start_frames)} estimated tracks were born after frame 0 "
      f"({near_division} of them within 1 frame of a true division)")

🔍 **What just happened?** Plain one-to-one Hungarian linking has **no concept of one detection becoming two**: at the frame where a cell divides, the parent's single detection cannot map one-to-one onto both daughter detections, so the assignment matches one daughter to the parent's track and leaves the other daughter **unmatched** — which our pipeline then starts as a brand-new, lineage-blind track id. This is not a bug in the Hungarian algorithm; it's a limitation of modelling tracking as a sequence of independent *frame-to-frame* one-to-one assignments.

## 5 · From one-to-one assignment to lineages: gap closing, merging, splitting

The fix, introduced by Jaqaman et al. (2008, *Nature Methods* — the paper that first applied LAP to particle tracking in live-cell imaging), is to **not throw away unmatched detections** after the frame-to-frame step. Instead, the frame-to-frame links are first chained into short *track segments*, and a **second LAP** is solved over whole segments, with an enlarged cost matrix that includes extra blocks for:

- **gap closing** — linking the end of one segment to the start of another a few frames later (closing a brief disappearance);
- **merging** — linking a segment's end to the *middle* of another segment (two objects becoming one, e.g. two overlapping cells);
- **splitting** — linking the *middle* of a segment to the start of a new one (one object becoming two — exactly our division case).

Each of these possible events gets its own cost, and a configurable "cost of doing nothing" lets the solver decide that some detections are genuinely better left unlinked. This two-stage LAP (particle-to-particle, then segment-to-segment) is still solved efficiently by the same Hungarian machinery on a larger (but still polynomial-size) cost matrix — no reformulation into a dramatically harder optimisation problem is needed.

### 5.1 Implementing the fix: a segment-level LAP

The description above has three moving pieces: short *track segments* built exactly the way Section 4.2 already does, a *second* cost matrix over whole segments (the gap-closing and splitting costs), and a *reject* ("cost of doing nothing") block so the solver can leave a segment unlinked rather than force a bad match. All three reuse code already in this notebook — `hungarian_assignment` is the same function from the 🧰 toolkit, and the frame-to-frame step is exactly the loop from 4.2, now wrapped as a function so it can be called again.


In [ ]:
# ── 5.1 The frame-to-frame linker from 4.2, wrapped so it can be reused ──
def track_by_detection(detections_per_frame, GATE=15.0):
    """Exactly the frame-to-frame Hungarian linking loop from 4.2, as a function: link each
    frame's detections to the previous frame's active tracks, gated at GATE px; every
    unmatched detection starts a new track id (this is where a division shows up as a brand
    new, parent-less track)."""
    next_track_id = 0
    active = {}
    estimated_tracks = {}
    for t, dets in enumerate(detections_per_frame):
        if not active:
            for y, x in dets:
                active[next_track_id] = (y, x)
                estimated_tracks[next_track_id] = [(t, y, x)]
                next_track_id += 1
            continue
        active_ids = list(active)
        prev_pts = np.array([active[i] for i in active_ids])
        if len(dets) == 0:
            active = {}
            continue
        cost = np.hypot(prev_pts[:, 0:1] - dets[:, 0], prev_pts[:, 1:2] - dets[:, 1])
        rows, cols = hungarian_assignment(cost)
        matched_dets = set()
        new_active = {}
        for r, c in zip(rows, cols):
            if cost[r, c] <= GATE:
                tid = active_ids[r]
                new_active[tid] = tuple(dets[c])
                estimated_tracks[tid].append((t, *dets[c]))
                matched_dets.add(c)
        for c in range(len(dets)):
            if c not in matched_dets:
                new_active[next_track_id] = tuple(dets[c])
                estimated_tracks[next_track_id] = [(t, *dets[c])]
                next_track_id += 1
        active = new_active
    return estimated_tracks


def segment_level_lap(estimated_tracks, GAP_FRAMES=2, SPLIT_GATE=20.0):
    """The second LAP: for every segment that starts after frame 0 (every "new, parent-less
    track" from 4.2/4.3), find the best candidate parent segment to link it to.

    - A candidate parent's cost is the distance between the new segment's first point and
      the parent's own most recent point in the GAP_FRAMES frames before it -- the
      gap-closing / splitting cost described above (a splitting link if the parent segment
      keeps going afterwards too, a gap-closing link if it does not).
    - A diagonal block of cost SPLIT_GATE is the "cost of doing nothing": linking a new
      segment to ITSELF at that cost is always an option, so the solver only proposes a real
      link when it is cheaper than just leaving the segment alone.
    """
    new_segments = [tid for tid, pts in estimated_tracks.items() if pts[0][0] > 0]
    parent_candidates = list(estimated_tracks.keys())
    n, m = len(new_segments), len(parent_candidates)
    if n == 0:
        return {}
    LARGE = 1e6
    cost = np.full((n, m), LARGE)
    for i, child in enumerate(new_segments):
        t0, y0, x0 = estimated_tracks[child][0]
        for j, parent in enumerate(parent_candidates):
            if parent == child:
                continue
            nearby = [(t, y, x) for (t, y, x) in estimated_tracks[parent] if 0 <= t0 - t <= GAP_FRAMES]
            if nearby:
                t, y, x = min(nearby, key=lambda p: t0 - p[0])  # parent's most recent point before the gap
                cost[i, j] = np.hypot(y - y0, x - x0)
    reject_block = np.full((n, n), LARGE)
    np.fill_diagonal(reject_block, SPLIT_GATE)
    full_cost = np.hstack([cost, reject_block])
    rows, cols = hungarian_assignment(full_cost)
    split_links = {}  # child segment id -> parent segment id
    for r, c in zip(rows, cols):
        if c < m and full_cost[r, c] < SPLIT_GATE:
            split_links[new_segments[r]] = parent_candidates[c]
    return split_links


### 5.2 A small movie where every link can be checked by eye

Section 4.2's ten-particle movie is realistic, but busy — several particles cross paths and a few divisions happen close together, which makes it hard to check any *one* link by eye. Before turning the fix loose on that busier movie, let's see it work on a smaller one.


In [ ]:
# ── 5.2 A smaller, less crowded movie -- every lineage can be checked by eye as well as against the ground truth ──
small_sim = simulate_time_lapse(n_frames=18, n_particles=6, size=160, speed=2.5,
                                 division_prob=0.04, seed=6)
small_detections = [detect_frame_blobs(f) for f in small_sim["frames"]]
small_estimated = track_by_detection(small_detections)
small_splits = segment_level_lap(small_estimated)

print(f"Ground truth : {len(small_sim['tracks'])} true track ids, {len(small_sim['lineage'])} division event(s)")
print(f"4.2's naive linker: {len(small_estimated)} estimated track ids "
      f"({sum(1 for v in small_estimated.values() if v[0][0] > 0)} of them new, parent-less tracks)")
print("5.1's segment-level LAP recovered these child -> parent links:", small_splits)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.6))
plot_tracks(axes[0], small_sim["tracks"], min_len=2)
axes[0].set_title("Ground truth tracks\n(colour = true track id)")
plot_tracks(axes[1], small_estimated, min_len=2)
axes[1].set_title("4.2's naive linking\n(every division makes a new colour)")
plot_tracks(axes[2], small_estimated, min_len=2)
for child, parent in small_splits.items():
    p0 = small_estimated[parent][-1]
    c0 = small_estimated[child][0]
    axes[2].plot([p0[2], c0[2]], [p0[1], c0[1]], "k--", linewidth=1.5)
axes[2].set_title("5.1's segment-level LAP\n(dashed line = recovered parent -> child link)")
fig.tight_layout()


In [ ]:
# ── 🧪 Ground-truth check: does every recovered link match a real division? ──
def label_segment_by_truth(pts, true_tracks, tol=6.0):
    """Label an estimated segment with whichever true track id its points agree with most
    often (nearest within `tol` px at each shared frame) -- a majority vote, used only to
    check our work against the hidden ground truth, never by the tracker itself."""
    votes = {}
    true_points = {tid: {t: (y, x) for t, y, x in tpts} for tid, tpts in true_tracks.items()}
    for t, y, x in pts:
        best_tid, best_d = None, tol
        for tid, tp in true_points.items():
            if t in tp:
                d = np.hypot(tp[t][0] - y, tp[t][1] - x)
                if d < best_d:
                    best_d, best_tid = d, tid
        if best_tid is not None:
            votes[best_tid] = votes.get(best_tid, 0) + 1
    return max(votes, key=votes.get) if votes else None

small_est_to_true = {tid: label_segment_by_truth(pts, small_sim["tracks"]) for tid, pts in small_estimated.items()}
correct, wrong = 0, 0
for child, parent in small_splits.items():
    child_true, parent_true = small_est_to_true.get(child), small_est_to_true.get(parent)
    really_is_a_division = small_sim["lineage"].get(child_true) == parent_true
    print(f"  recovered link: segment {child} (true track {child_true}) <- segment {parent} "
          f"(true track {parent_true})  ->  {'a real division' if really_is_a_division else 'WRONG'}")
    correct += really_is_a_division
    wrong += not really_is_a_division
n_lineage = len(small_sim["lineage"])
print(f"\n{correct} of {n_lineage} true divisions correctly recovered, {wrong} wrong link(s)")
print("(4.2's naive one-to-one linking recovers 0 of these by construction: every division "
      "becomes a brand-new, parent-less track id.)")
assert correct == n_lineage and wrong == 0, \
    "the segment-level LAP should recover every division on this small, uncrowded movie"


### 5.3 Scaling up: the same fix on 4.2's busier movie

Applying `segment_level_lap` to 4.2's own ten-particle `estimated_tracks` (no need to re-simulate — it is still in memory) is one line. The result on a busier movie is more honest, and more useful to know going in: linking only ever sees what the detector handed it.


In [ ]:
# ── 5.3 The same fix, unchanged, on 4.2's busier ten-particle movie ──
busy_splits = segment_level_lap(estimated_tracks)
busy_est_to_true = {tid: label_segment_by_truth(pts, sim["tracks"]) for tid, pts in estimated_tracks.items()}
busy_true_to_est = {}
for tid, true_id in busy_est_to_true.items():
    busy_true_to_est.setdefault(true_id, []).append(tid)

never_detected = correct = recoverable = 0
for child_true, parent_true in sim["lineage"].items():
    child_segments = busy_true_to_est.get(child_true, [])
    if not child_segments:
        never_detected += 1
        continue
    recoverable += 1
    if any(busy_splits.get(seg) is not None and busy_est_to_true.get(busy_splits[seg]) == parent_true
           for seg in child_segments):
        correct += 1

print(f"{len(sim['lineage'])} true division events in 4.2's busier movie")
print(f"{never_detected} were never resolved as a separate blob at all by detect_frame_blobs "
      "(the two daughters stayed too close together) -- no linking algorithm can recover an "
      "object the detector already merged away")
print(f"{correct} of the remaining {recoverable} were correctly linked back to their true parent")
print("Busier movies make the segment-level LAP's job harder in two separate ways: more "
      "detections to confuse a nearby but wrong candidate with, and more crowding for the "
      "detector itself -- a reminder that tracking is a pipeline, and fixing the linking "
      "stage cannot undo a mistake the detection stage already made.")


📘 **Lecture notes — beyond the segment-level LAP.** Three further families are worth knowing by name, even without implementing them here. **Multiple hypothesis tracking (MHT)** keeps *several* candidate linkings alive simultaneously as a tree of hypotheses, deferring the final decision until more frames of evidence have accumulated — more reliable in ambiguous situations, but combinatorially expensive without aggressive pruning (e.g. "N-scan pruning," which keeps only hypotheses sharing the globally-best branch a few frames back). **Global track linking via dynamic programming** (e.g. Magnusson and Jaldén's Viterbi-based linker) reformulates the whole-movie linking decision as a shortest-path problem solved once over every frame simultaneously, guaranteeing a globally optimal set of tracks under its cost model rather than the segment-level LAP's locally-optimal two-stage compromise. **Deep-learning trackers** — both general ones (an MLP appearance feature feeding an LSTM/recurrent motion model) and ones built specifically for bioimaging (e.g. EmbedTrack, Trackastra, btrack) — learn the association and/or motion-prediction step directly from labelled tracking data instead of hand-designing a cost function, at the price of needing that labelled data.


## 6 · Tracking without detection: optical flow for collective motion

Everything so far has shared one assumption: there is a *discrete object* to detect every frame, and tracking means linking those detections across time. Many bioimaging problems do not look like that at all — a sheet of epithelial cells shearing past each other, cortical actomyosin flowing beneath a cell membrane, cytoplasmic streaming — there is no single blob to pick out, only a continuously textured image that *moves*. **Optical flow** estimates that motion directly from the images themselves, with no detection step at all: for every pixel (or every point you choose to query), it asks how the image's own local texture has to shift to explain what changed between two frames.


### 6.1 Lucas-Kanade optical flow, from scratch

Assume the image's intensity at any point only ever *moves*, never changes (the **brightness constancy** assumption): `I(y + v_y, x + v_x, t + 1) = I(y, x, t)`. A first-order Taylor expansion turns this into one linear equation per pixel relating the unknown flow `(v_y, v_x)` to the image's spatial gradients `(I_y, I_x)` and its time derivative `I_t`:

`I_y · v_y + I_x · v_x = -I_t`

One equation, two unknowns — underdetermined at a single pixel (the **aperture problem**, §6.2 below). Lucas and Kanade's (1981) fix: assume the flow is *constant* over a small window, stack every pixel in the window into an overdetermined linear system, and solve it by least squares — the normal equations reduce to a 2×2 system.


In [ ]:
# ── 6.1 A from-scratch Lucas-Kanade sparse optical-flow estimator ──
def simulate_speckle_pattern(size=160, n_specks=500, speck_sigma=1.6, seed=0):
    """A SIMULATED textured 'speckle' field: many small random-intensity blobs, so every
    local window has distinctive, non-uniform texture for optical flow to lock onto
    (mimicking a speckled/textured tissue or a bead-seeded gel used for PIV, never a
    specific real image)."""
    rng = np.random.default_rng(seed)
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    canvas = np.zeros((size, size))
    for y, x, a in zip(rng.uniform(0, size, n_specks), rng.uniform(0, size, n_specks),
                        rng.uniform(0.5, 1.0, n_specks)):
        canvas += a * np.exp(-(((yy - y) ** 2 + (xx - x) ** 2) / (2 * speck_sigma ** 2)))
    return canvas


def warp_by_flow(image, flow_fn):
    """Build the NEXT frame under a KNOWN analytic flow field (y, x) -> (v_y, v_x):
    backward-warp (sample the image at position - v) so the forward flow v truly maps
    image -> warped image."""
    size = image.shape[0]
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    vy, vx = flow_fn(yy, xx)
    warped = ndi.map_coordinates(image, [yy - vy, xx - vx], order=1, mode="reflect")
    return warped, (vy, vx)


def lucas_kanade_flow(frame0, frame1, points, window=13):
    """The Lucas-Kanade estimate of (v_y, v_x) at each (y, x) in `points`, from a
    `window`x`window` neighbourhood of image gradients. Returns NaN where the 2x2 system is
    (near-)singular -- the aperture problem, see 6.2."""
    Iy, Ix = np.gradient(frame0)
    It = frame1 - frame0
    r = window // 2
    flows = np.full((len(points), 2), np.nan)
    for i, (y, x) in enumerate(points):
        y, x = int(round(y)), int(round(x))
        iy = Iy[max(0, y - r):y + r + 1, max(0, x - r):x + r + 1].ravel()
        ix = Ix[max(0, y - r):y + r + 1, max(0, x - r):x + r + 1].ravel()
        it = It[max(0, y - r):y + r + 1, max(0, x - r):x + r + 1].ravel()
        A = np.stack([iy, ix], axis=1)
        AtA = A.T @ A
        if np.linalg.cond(AtA) > 1e8:
            continue  # the window is too uniform to pin down both components -- see 6.2
        flows[i] = np.linalg.solve(AtA, -A.T @ it)
    return flows


# A known, simple analytic flow: horizontal shear (as if a textured sheet were sliding past
# itself), centred on the image -- cortical/epithelial shear flows look much like this locally.
SHEAR_K = 0.02  # px of v_x per px of (y - centre), per frame
flow_size = 160
flow_centre = flow_size / 2

def shear_flow(yy, xx):
    return np.zeros_like(yy, dtype=float), SHEAR_K * (yy - flow_centre)

flow_frame0 = simulate_speckle_pattern(size=flow_size, seed=SEED)
flow_frame1, (vy_true_field, vx_true_field) = warp_by_flow(flow_frame0, shear_flow)

flow_grid = np.array([(y, x) for y in range(20, flow_size - 20, 12)
                       for x in range(20, flow_size - 20, 12)], dtype=float)
flow_est = lucas_kanade_flow(flow_frame0, flow_frame1, flow_grid, window=13)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
show_image(axes[0], flow_frame0, "frame 0 (SIMULATED speckle texture)")
show_image(axes[1], flow_frame1, "frame 1 (sheared by a KNOWN flow field)")
flow_valid = ~np.isnan(flow_est[:, 0])
for ax in axes:
    ax.quiver(flow_grid[flow_valid, 1], flow_grid[flow_valid, 0],
              flow_est[flow_valid, 1], flow_est[flow_valid, 0],
              color="red", angles="xy", scale_units="xy", scale=0.2, width=0.004)
fig.tight_layout()


In [ ]:
# ── 🧪 Ground-truth check: estimated flow vs. the known analytic shear field ──
flow_true_vy = np.zeros(len(flow_grid))
flow_true_vx = SHEAR_K * (flow_grid[:, 0] - flow_centre)
flow_err = np.hypot(flow_est[flow_valid, 0] - flow_true_vy[flow_valid],
                     flow_est[flow_valid, 1] - flow_true_vx[flow_valid])
print(f"{flow_valid.sum()} of {len(flow_grid)} query points gave a solvable flow estimate")
print(f"mean flow error: {flow_err.mean():.3f} px/frame "
      f"(true |v_x| ranges {np.abs(flow_true_vx).min():.2f}-{np.abs(flow_true_vx).max():.2f} px/frame)")
print(f"max flow error:  {flow_err.max():.3f} px/frame")
assert flow_valid.all(), "every query point should sit in enough speckle texture to be solvable"
assert flow_err.mean() < 0.3, "the recovered flow should track the known shear field closely"


### 6.2 ✏️ The aperture problem

Lucas-Kanade's 2×2 system is only solvable when the window contains texture in *at least two different directions* — otherwise `AᵀA` is singular (or close to it) and there is no unique `(v_y, v_x)`, only a one-dimensional family of flows that all fit the window's single gradient direction equally well. A perfectly uniform patch is the most extreme case: every gradient is zero, so even that one equation vanishes and literally any flow fits. Check this directly:


In [ ]:
# ── A uniform (textureless) patch: no gradient in any direction, so no flow can be recovered ──
blank0 = np.zeros((40, 40))
blank1, _ = warp_by_flow(blank0, lambda yy, xx: (np.zeros_like(yy, dtype=float),
                                                   np.full_like(xx, 2.0, dtype=float)))
flat_estimate = lucas_kanade_flow(blank0, blank1, [(20, 20)], window=13)
print("True flow applied to the blank patch: (0.00, 2.00) px/frame")
print("Lucas-Kanade's estimate on that uniform patch:", flat_estimate[0],
      " -- unsolvable: no texture, no gradient, no information")
assert np.isnan(flat_estimate[0]).all(), "a perfectly uniform window has no gradient to estimate flow from"


📘 **Lecture notes — optical flow in context.** Lucas and Kanade (1981) assumes flow is locally constant over a small window and solves a sparse, independent 2×2 system per point — cheap, and exactly what we implemented above. Horn and Schunck (1981) instead solves for a *dense*, globally smooth flow field over the whole image at once, trading Lucas-Kanade's local independence for a smoothness penalty between neighbouring pixels — better in low-texture regions, at the cost of a much larger (though still linear) system. Farnebäck (2003) is a practical, fast dense method (the one behind most ready-made "optical flow" functions you will meet in a library) based on local polynomial expansions rather than the brightness-constancy gradient directly. In bioimaging, exactly this kind of pixel-wise flow — rather than detection-and-linking — is how collective tissue motion gets quantified: Petitjean et al. (2010) applied PIV (particle image velocimetry, optical flow's close cousin) to measure velocity fields during epithelial morphogenesis, and Mayer et al. (2010) used cortical flow fields to show how their directionality helps position the first cleavage furrow in the early *C. elegans* embryo. The two tracking paradigms this notebook covers are genuinely complementary, not competing: detection-and-linking (Sections 1–5) is the right tool when there are discrete, nameable objects — cells, nuclei, particles — even if they occasionally divide or disappear; optical flow (this section) is the right tool when the motion is continuous and collective and there is nothing discrete to detect in the first place, as long as the per-frame displacement stays small enough for the brightness-constancy linearisation to hold.


## 📌 Key takeaways

- Per-frame detection alone cannot answer "which object became which" — tracking is specifically the business of recovering that correspondence over time.
- **Contour-evolution tracking** reuses the previous frame's converged boundary as the next frame's initial curve, letting a snake "follow" a slowly-deforming object such as a growing actin filament.
- **Particle filters** represent an object's state as a weighted cloud of hypotheses and update it by a predict/weight/resample cycle derived from Bayes' rule; the **Kalman filter** is the exact, cheap special case for linear-Gaussian dynamics.
- **Tracking-by-detection** decouples detection from linking and poses linking as a **linear assignment problem**, solved exactly and efficiently by the **Hungarian algorithm**.
- A single frame-to-frame LAP cannot represent divisions or merges; a second, **segment-level LAP** (Jaqaman et al. 2008) with extra cost blocks for gap-closing and splitting — implemented in Section 5 — recovers much of that lost lineage information, limited mainly by what the upstream detector resolved as separate objects in the first place.
- **Optical flow** (Lucas-Kanade, Section 6) tracks *motion* directly from image gradients, with no detection step at all — the right tool for continuous, collective movement (tissue shear, cortical flow) rather than discrete, nameable objects, as long as the per-frame displacement stays small and the local texture is not too uniform (the **aperture problem**).


## ✅ Checkpoint

1. Why can't an object detector, run independently on every frame, give you object velocities?
   <details><summary>Answer</summary>A detector only looks at one frame at a time; velocity requires knowing which detection in frame <i>t</i> corresponds to which detection in frame <i>t+1</i>, which a single-frame detector has no mechanism to determine.</details>
2. In a particle filter, what do the "weight" and "resample" steps each accomplish?
   <details><summary>Answer</summary>Weighting scores each particle by how well it explains the newest measurement (higher weight for particles closer to the true state); resampling draws a new particle cloud proportional to those weights, concentrating future computation on the more probable hypotheses and discarding unlikely ones.</details>
3. Why is the Hungarian algorithm needed instead of just greedily linking each detection to its nearest neighbour in the next frame?
   <details><summary>Answer</summary>Greedy nearest-neighbour linking can be globally suboptimal and can assign two different track detections to the same new detection; the Hungarian algorithm finds the assignment that minimises <i>total</i> cost over <i>all</i> links simultaneously, subject to the one-to-one constraint.</details>
4. Why does a plain frame-to-frame LAP "lose" lineage information at a cell division?
   <details><summary>Answer</summary>One-to-one assignment cannot map a single parent detection onto two daughter detections in the next frame; one daughter gets linked and the other is left unmatched, starting a lineage-blind new track, unless the cost matrix is explicitly extended with a splitting block (as in Jaqaman et al. 2008 and implemented in Section 5).</details>
5. What's the key difference between the segment-level LAP's "cost of doing nothing" and simply lowering the `GATE` threshold in the frame-to-frame step?
   <details><summary>Answer</summary>Lowering GATE makes every link harder to form everywhere at once, discarding good links along with the unwanted ones; the reject block only discourages one specific bad link (a given child segment to a given candidate parent) while the same cost matrix is free to accept good links elsewhere -- a per-pair option, not a global threshold.</details>
6. Why does Lucas-Kanade need a textured window rather than, in principle, a single pixel?
   <details><summary>Answer</summary>The brightness-constancy equation at one pixel is one linear equation in two unknowns (v_y, v_x) -- underdetermined (the aperture problem). A window spanning gradients in more than one direction supplies enough independent equations to pin down both components by least squares.</details>


## 🧠 Questions & discussion

- The Clark–Evans-style "gating threshold" (`GATE` in Section 4.2) treats any link costing more than 15 px as impossible. What biological or imaging scenario would make this threshold too strict, and what would too strict a gate do to your tracks?
- A Kalman filter assumes constant velocity between measurements. Sketch (in words) a cell behaviour for which this assumption would be badly wrong, and describe what change to the filter (not necessarily implemented here) would help.
- Suppose two cells touch and partially overlap for exactly one frame, then separate. Which of the LAP extensions in Section 5 (gap closing / merging / splitting) would be needed to track them correctly through that frame, and why?
- Section 5.3 found that most of the ten-particle movie's missed divisions were never even detected as two separate blobs. Would widening `GAP_FRAMES` or `SPLIT_GATE` in `segment_level_lap` fix that, or is a different part of the pipeline responsible? Which one, and why?
- An epithelial sheet is sliding past a fixed scaffold at a slow, steady rate, with no individual cell boundaries resolvable. Would you reach for tracking-by-detection or optical flow to measure its speed, and why?


## 🚀 Bonus — going further

- Add a simple appearance feature (e.g. mean intensity, or `shape_features` from Module 15) to the linking cost matrix alongside distance, and see whether it reduces identity switches when two tracks pass close to each other.
- Extend `segment_level_lap`'s cost matrix with a *merging* block (linking a segment's end to the middle of another segment, not just its start) and test it on a modified `simulate_time_lapse` call where two particles are made to pass close enough to briefly merge into one detection.
- Compare the particle filter's accuracy against the Kalman filter's as you make the simulated motion less linear (e.g. add a sinusoidal wiggle to the particle's velocity in a modified `simulate_time_lapse`-style generator).
- Make `lucas_kanade_flow` dense instead of sparse (run it on every pixel, not a grid of query points) and visualise the full flow field as a colour-wheel image (hue = direction, brightness = magnitude) instead of a quiver plot.


## 📝 Lab note

```text
Module 22 — Object tracking
Data: SIMULATED dividing time-lapse, course_toolkit.simulate_time_lapse(seed=14)
Pipeline: blob detection (threshold + connected components) -> Hungarian frame-to-frame
          linking (gate = 15 px) -> track segments -> segment-level LAP (gap closing/splitting)
Result: [ ] estimated track ids vs [ ] true track ids ([ ] true divisions)
Particle filter mean position error: [ ] px | Kalman filter mean position error: [ ] px
Segment-level LAP (small movie): [ ] of [ ] true divisions recovered
Segment-level LAP (busier movie): [ ] of [ ] recoverable divisions recovered ([ ] never detected)
Lucas-Kanade optical flow: mean error [ ] px/frame, [ ] of [ ] query points solvable
Notes:
```


## 📚 References

1. Jaqaman, K. et al. (2008). Robust single-particle tracking in live-cell time-lapse sequences. *Nature Methods*.
2. Kuhn, H. W. (1955). The Hungarian method for the assignment problem. *Naval Research Logistics Quarterly*.
3. Kass, M., Witkin, A., & Terzopoulos, D. (1988). Snakes: Active contour models. *International Journal of Computer Vision*.
4. Li, H., Shen, T., Vavylonis, D., & Huang, X. (2009). Actin filament tracking based on particle filters and stretching open active contour models. *MICCAI*.
5. Hadjidemetriou, S., Toomre, D., & Duncan, J. (2008). Motion tracking of the outer tips of microtubules. *Medical Image Analysis*.
6. Doucet, A., de Freitas, N., & Gordon, N. (2001). *Sequential Monte Carlo Methods in Practice* (particle filters).
7. Kalman, R. E. (1960). A new approach to linear filtering and prediction problems. *Journal of Basic Engineering*.
8. Veenman, C. J., Reinders, M. J. T., & Backer, E. (2001). Resolving motion correspondence for densely moving points. *IEEE PAMI* (multiple hypothesis tracking context).
9. Maška, M. et al. (2014). A benchmark for comparison of cell tracking algorithms. *Bioinformatics*.
10. Lucas, B. D., & Kanade, T. (1981). An iterative image registration technique with an application to stereo vision. *IJCAI*.
11. Horn, B. K. P., & Schunck, B. G. (1981). Determining optical flow. *Artificial Intelligence*.
12. Farnebäck, G. (2003). Two-frame motion estimation based on polynomial expansion. *Image Analysis (SCIA)*.
13. Petitjean, L. et al. (2010). Velocity fields in a collectively migrating epithelium. *Biophysical Journal*.
14. Mayer, M., Depken, M., Bois, J. S., Jülicher, F., & Grill, S. W. (2010). Anisotropies in cortical tension reveal the physical basis of polarizing cortical flows. *Nature*.


## ➡️ Next

[**Module 23 — Image registration**](23_image_registration.ipynb): once you can track one object through time, the natural next question is how to align two *whole images* — of the same sample at different times, from different modalities, or from different individuals — into a shared coordinate system.